# Daily Offers — Revenue Performance

In [1]:
# hide-output

# show → code input visible by default
# hide-output → output hidden by default
# show hide-output → both (can combine on one line)

import pandas as pd
import plotly.express as px
#from google.cloud import bigquery
from common_lib.sql import BigQueryConnector
from common_lib.export import export_notebook_html

In [2]:
# Single flag controls every BQ pull in this notebook (revenue, checkout starts, rewards) via
# bqc.load_or_query() below — keeps all three datasets on the same as-of date instead of drifting
# out of sync.
refresh_data = False
bqc = BigQueryConnector()

In [3]:
# hide-output
data = bqc.load_or_query('daily_offers', refresh_data, query_parameters={'period_offset': 365})

This query will process 7.5 MB when run.
Estimated query cost: $0.00


## Raw Data

In [4]:
#data

## Product-Level Aggregation

`data` is now at user–day–product grain, so `first_seen`/`last_seen` in the raw rows are just each row's own `dt` — not useful on their own. Roll up to `product_summary` (one row per offer) to get real first/last-seen dates, `active_days`, and buyer counts; all the charts below aggregate from `data` or `product_summary` rather than assuming pre-aggregated rows.

`active_days` normalizes across offers that have been live for different lengths of time (e.g. new 2026Q3 offers vs. 2025Q4 offers that have run the full window) — `revenue_per_day` is the fairer basis for ranking offer performance than raw total revenue.

In [5]:
data['offer_quarter'] = data['product_id'].str.extract(r'(\d{4}Q\d)')
data['bundle_type'] = data['product_id'].str.extract(r'-[A-Za-z]+-([A-Za-z]+)(?:\s[\d.]+)?$')
data['product_price_group'] = data['product_price_group'].str.replace('daily_offers_', '', regex=False)

product_summary = data.groupby(
    ['product_id', 'product_type', 'product_theme', 'price_point_usd', 'product_price_group',
     'offer_quarter', 'bundle_type'],
    as_index=False,
).agg(
    first_seen=('dt', 'min'),
    last_seen=('dt', 'max'),
    n_trans=('n_trans', 'sum'),
    n_buyers=('user_id', 'nunique'),
    usd_revenue=('usd_revenue', 'sum'),
)

product_summary['active_days'] = (product_summary['last_seen'] - product_summary['first_seen']).dt.days + 1
product_summary['revenue_per_day'] = (product_summary['usd_revenue'] / product_summary['active_days']).round(2)
product_summary['revenue_per_trans'] = (product_summary['usd_revenue'] / product_summary['n_trans']).round(2)
product_summary['revenue_share_pct'] = (product_summary['usd_revenue'] / product_summary['usd_revenue'].sum() * 100).round(2)

#product_summary.sort_values('usd_revenue', ascending=False)

## Revenue by Product Type

In [6]:
# Fixed identity color per product_type, kept stable regardless of ranking
type_colors = {
    'RotatingIncPack': '#2a78d6',
    'Rotating': '#eb6834',
    'NonPayerIncPack': '#1baf7a',
    'NonPayer': '#eda100',
}

by_type = product_summary.groupby('product_type', as_index=False).agg(
    usd_revenue=('usd_revenue', 'sum'),
    n_trans=('n_trans', 'sum'),
    n_offers=('product_id', 'nunique'),
).sort_values('usd_revenue', ascending=False)

fig = px.bar(
    by_type, x='product_type', y='usd_revenue', color='product_type',
    color_discrete_map=type_colors,
    custom_data=['n_trans', 'n_offers'],
)
fig.update_traces(
    hovertemplate='Revenue: $%{y:,.0f}<br>Transactions: %{customdata[0]:,.0f}<br>Offers: %{customdata[1]}<extra></extra>'
)
fig.update_layout(
    title='Revenue by Product Type',
    xaxis=dict(title=None),
    yaxis=dict(title='Revenue (USD)', tickprefix='$'),
    height=450, width=800,
    showlegend=False,
)
fig.show()

## Revenue by Bundle Type

In [7]:
# Fixed identity color per bundle_type, kept stable regardless of ranking
bundle_colors = {
    'CafeBundle': '#2a78d6',
    'GardenBundle': '#eb6834',
    'PartyBundle': '#1baf7a',
    'BundleSmall': '#eda100',
    'BundleMedium': '#e87ba4',
    'BundleLarge': '#008300',
}

by_bundle = product_summary.groupby(['bundle_type','offer_quarter'], as_index=False).agg(
    usd_revenue=('usd_revenue', 'sum'),
    n_trans=('n_trans', 'sum'),
    n_offers=('product_id', 'nunique'),
).sort_values('usd_revenue', ascending=False)

fig = px.bar(
    by_bundle, x='bundle_type', y='usd_revenue', color='offer_quarter',
    color_discrete_map=bundle_colors,
    custom_data=['n_trans', 'n_offers'],
)
fig.update_traces(
    hovertemplate='Revenue: $%{y:,.0f}<br>Transactions: %{customdata[0]:,.0f}<br>Offers: %{customdata[1]}<extra></extra>'
)
fig.update_layout(
    title='Revenue by Bundle Type',
    xaxis=dict(title=None),
    yaxis=dict(title='Revenue (USD)', tickprefix='$'),
    height=450, width=800,
    showlegend=False,
)
fig.show()

## Revenue by Price Point

A quick look at volumes both in revenue and transactions, gives glimpse of what pricepoints the new offers were located at and where they caused more impact

In [8]:
by_price = product_summary.groupby(['price_point_usd','offer_quarter'], as_index=False).agg(
    usd_revenue=('usd_revenue', 'sum'),
    n_trans=('n_trans', 'sum'),
    n_offers=('product_id', 'nunique'),
).sort_values('price_point_usd')
by_price = by_price.assign(price_label=by_price['price_point_usd'].astype(str))

fig = px.bar(
    by_price, 
    x='price_label', 
    y='usd_revenue',
    color='offer_quarter',
    custom_data=['n_trans', 'n_offers'],
)
fig.update_traces(
    #marker_color='steelblue',
    hovertemplate='Price: $%{x}<br>Revenue: $%{y:,.0f}<br>Transactions: %{customdata[0]:,.0f}<br>Offers: %{customdata[1]}<extra></extra>'
)
fig.update_layout(
    title='Revenue by Price Point',
    xaxis=dict(title='Price (USD)', type='category'),
    yaxis=dict(title='Revenue (USD)', tickprefix='$'),
    height=450, width=1000,
)
fig.show()

fig = px.bar(
    by_price, 
    x='price_label', 
    y='n_trans',
    color='offer_quarter',
    custom_data=['n_trans', 'n_offers'],
)
fig.update_traces(
    #marker_color='steelblue',
    hovertemplate='Price: $%{x}<br>Transactions: %{y:,.0f}<br>Transactions: %{customdata[0]:,.0f}<br>Offers: %{customdata[1]}<extra></extra>'
)
fig.update_layout(
    title='Transactions by Price Point',
    xaxis=dict(title='Price (USD)', type='category'),
    yaxis=dict(title='Transactions', tickprefix=''),
    height=450, width=1000,
)
fig.show()

## Revenue by Offer Quarter

Raw totals — 2026Q3 offers have only been live a few days vs. the full window for 2025Q4, so this isn't a fair per-offer comparison (see the normalized ranking below).

In [9]:
by_quarter = product_summary.groupby('offer_quarter', as_index=False).agg(
    usd_revenue=('usd_revenue', 'sum'),
    n_trans=('n_trans', 'sum'),
    n_offers=('product_id', 'nunique'),
).sort_values('offer_quarter')

fig = px.bar(
    by_quarter, x='offer_quarter', y='usd_revenue',
    custom_data=['n_trans', 'n_offers'],
)
fig.update_traces(
    marker_color='steelblue',
    hovertemplate='Revenue: $%{y:,.0f}<br>Transactions: %{customdata[0]:,.0f}<br>Offers: %{customdata[1]}<extra></extra>'
)
fig.update_layout(
    title='Revenue by Offer Quarter',
    xaxis=dict(title=None, type='category'),
    yaxis=dict(title='Revenue (USD)', tickprefix='$'),
    height=450, width=800,
)
fig.show()

In the daily revenue and transactions we see the first time of underperformance in the number of transactions, being the lowest one on this period for daily offers

In [10]:
daily_by_quarter = data.groupby(['dt', 'offer_quarter'], as_index=False).agg(
    usd_revenue=('usd_revenue', 'sum'),
    n_trans=('n_trans', 'sum'),
    avg_usd_revenue=('usd_revenue', 'mean'),
    avg_price_point_usd=('price_point_usd', 'mean')
).sort_values('dt')

quarter_colors = {'2025Q4': '#2a78d6', '2026Q3': '#eb6834'}

# Daily revenue per transaction by day and quarter

fig = px.bar(
    daily_by_quarter, x='dt', y='usd_revenue', color='offer_quarter',
    color_discrete_map=quarter_colors,
    custom_data=['n_trans'],
)
fig.update_traces(
    #mode='lines+markers',
    hovertemplate='Date: %{x|%Y-%m-%d}<br>Revenue: $%{y:,.0f}<br>Transactions: %{customdata[0]:,.0f}<extra></extra>',
)
fig.update_layout(
    title='Daily Revenue by Offer Quarter',
    xaxis=dict(title='Date'),
    yaxis=dict(title='Revenue (USD)', tickprefix='$'),
    height=450, width=1100,
    legend=dict(orientation='h', yanchor='bottom', y=-0.3, xanchor='center', x=0.5, title=None),
)
fig.show()

# Daily transactions by day and quarter

fig = px.bar(
    daily_by_quarter, 
    x='dt', 
    y='n_trans', 
    color='offer_quarter',
    color_discrete_map=quarter_colors,
    custom_data=['n_trans'],
)
fig.update_traces(
    #mode='lines+markers',
    hovertemplate='Date: %{x|%Y-%m-%d}<br>Transactions: %{y:,.0f}<br>Transactions: %{customdata[0]:,.0f}<extra></extra>',
)
fig.update_layout(
    title='Daily Transactions by Offer Quarter',
    xaxis=dict(title='Date'),
    yaxis=dict(title='Transactions'),
    height=450, width=1100,
    legend=dict(orientation='h', yanchor='bottom', y=-0.3, xanchor='center', x=0.5, title=None),
)
fig.show()

# Average revenue per transaction by day and quarter

fig = px.line(
    daily_by_quarter, x='dt', y='avg_usd_revenue', color='offer_quarter',
    color_discrete_map=quarter_colors,
    custom_data=['n_trans'],
)
fig.update_traces(
    mode='lines+markers',
    hovertemplate='Date: %{x|%Y-%m-%d}<br>Average Revenue: $%{y:,.2f}<br>Transactions: %{customdata[0]:,.0f}<extra></extra>',
)
fig.update_layout(
    title='Daily Average Revenue by Offer Quarter',
    xaxis=dict(title='Date'),
    yaxis=dict(title='Average Revenue (USD)', tickprefix='$'),
    height=450, width=1100,
    legend=dict(orientation='h', yanchor='bottom', y=-0.3, xanchor='center', x=0.5, title=None),
)
fig.show()



# Average price_point  by day and quarter

fig = px.line(
    daily_by_quarter, x='dt', y='avg_price_point_usd', color='offer_quarter',
    color_discrete_map=quarter_colors,
    custom_data=['n_trans'],
)
fig.update_traces(
    mode='lines+markers',
    hovertemplate='Date: %{x|%Y-%m-%d}<br>Average Price Point: $%{y:,.2f}<br>Transactions: %{customdata[0]:,.0f}<extra></extra>',
)
fig.update_layout(
    title='Daily Average Price Point by Offer Quarter',
    xaxis=dict(title='Date'),
    yaxis=dict(title='Average Price Point (USD)', tickprefix='$'),
    height=450, width=1100,
    legend=dict(orientation='h', yanchor='bottom', y=-0.3, xanchor='center', x=0.5, title=None),
)
fig.show()



## Revenue by Offer Quarter and Price point

In [11]:
daily_by_quarter = data.groupby(['dt', 'offer_quarter','price_point_usd'], as_index=False).agg(
    usd_revenue=('usd_revenue', 'sum'),
    n_trans=('n_trans', 'sum'),
    avg_usd_revenue=('usd_revenue', 'mean'),
).sort_values('dt')

daily_by_quarter['combined_dim'] = '$' + daily_by_quarter['price_point_usd'].astype(str) + ' · ' + daily_by_quarter['offer_quarter']

daily_by_quarter.sort_values(by=['dt', 'combined_dim'], inplace=True)

combined_dim_order = (
    daily_by_quarter[['price_point_usd', 'offer_quarter', 'combined_dim']]
    .drop_duplicates()
    .sort_values(['price_point_usd', 'offer_quarter'])['combined_dim']
    .tolist()
)

quarter_colors = {'2025Q4': '#2a78d6', '2026Q3': '#eb6834'}

fig = px.bar(
    daily_by_quarter, 
    x='dt', 
    y='usd_revenue', 
    color='combined_dim',
    category_orders={'combined_dim': combined_dim_order},
    #color_discrete_map=quarter_colors,
    custom_data=['n_trans','offer_quarter','price_point_usd'],
)
fig.update_traces(
    #mode='lines+markers',
    hovertemplate='Date: %{x|%Y-%m-%d}<br>Revenue: $%{y:,.0f}<br>Transactions: %{customdata[0]:,.0f}<br>Offer Quarter: %{customdata[1]}<br>Price Point: $%{customdata[2]:,.2f}<extra></extra>',
)
fig.update_layout(
    title='Daily Revenue by Offer Quarter and Price Point',
    xaxis=dict(title='Date'),
    yaxis=dict(title='Revenue (USD)', tickprefix='$'),
    height=450, 
    width=1100,
    legend=dict(orientation='v', yanchor='top', xanchor='right',title=None),
)
fig.show()

In [12]:
daily_by_quarter = data.groupby(['dt', 'offer_quarter','price_point_usd','product_price_group'], as_index=False).agg(
    usd_revenue=('usd_revenue', 'sum'),
    n_trans=('n_trans', 'sum'),
    avg_usd_revenue=('usd_revenue', 'mean'),
).sort_values('dt')

daily_by_quarter['combined_dim'] = '$' + daily_by_quarter['price_point_usd'].astype(str) + ' · ' + daily_by_quarter['offer_quarter']

daily_by_quarter.sort_values(by=['dt', 'combined_dim'], inplace=True)

combined_dim_order = (
    daily_by_quarter[['price_point_usd', 'offer_quarter', 'combined_dim']]
    .drop_duplicates()
    .sort_values(['price_point_usd', 'offer_quarter'])['combined_dim']
    .tolist()
)
price_tier_order = (
    daily_by_quarter.groupby('product_price_group')['price_point_usd'].min().sort_values().index.tolist()
)

quarter_colors = {'2025Q4': '#2a78d6', '2026Q3': '#eb6834'}

fig = px.bar(
    daily_by_quarter, 
    x='dt', 
    y='usd_revenue', 
    color='combined_dim',
    facet_row='product_price_group',
    category_orders={'combined_dim': combined_dim_order, 'product_price_group': price_tier_order},
    #color_discrete_map=quarter_colors,
    custom_data=['n_trans','offer_quarter','price_point_usd'],
)
fig.update_traces(
    #mode='lines+markers',
    hovertemplate='Date: %{x|%Y-%m-%d}<br>Revenue: $%{y:,.0f}<br>Transactions: %{customdata[0]:,.0f}<br>Offer Quarter: %{customdata[1]}<br>Price Point: $%{customdata[2]:,.2f}<extra></extra>',
)
fig.update_layout(
    title='Daily Revenue by Offer Quarter and Price Point',
    xaxis=dict(title='Date'),
    yaxis=dict(title='Revenue (USD)', tickprefix='$'),
    height=1400, 
    width=1100,
    legend=dict(orientation='v', yanchor='top', xanchor='right',x=-.2,title=None),
)
fig.show()

## Top Offers by Revenue per Active Day

Normalized ranking — controls for offers that have only been live a few days (e.g. new 2026Q3 variants) vs. offers that have run the full window. Restricted to offers with >= 2 active days, since single-day figures are too noisy to rank on.

In [13]:
top_n = 15

top_per_day = product_summary[product_summary['active_days'] >= 2].nlargest(top_n, 'revenue_per_day').sort_values('revenue_per_day')
top_per_day = top_per_day.assign(
    label=top_per_day['bundle_type'] + ' · ' + top_per_day['offer_quarter'] + ' · $' + top_per_day['price_point_usd'].astype(str)
)

fig = px.bar(
    top_per_day, x='revenue_per_day', y='label', orientation='h',
    custom_data=['usd_revenue', 'active_days', 'price_point_usd'],
)
fig.update_traces(
    marker_color='steelblue',
    hovertemplate=(
        'Revenue/day: $%{x:,.0f}<br>Total revenue: $%{customdata[0]:,.0f}<br>'
        'Active days: %{customdata[1]}<br>Price: $%{customdata[2]:.2f}<extra></extra>'
    ),
)
fig.update_layout(
    title=f'Top {top_n} Daily Offers by Revenue per Active Day',
    xaxis=dict(title='Revenue per Active Day (USD)', tickprefix='$'),
    yaxis=dict(title=None, automargin=True),
    height=550, width=1100,
    margin=dict(l=10),
)
fig.show()

## Checkout Conversion: Started vs Completed

`fact_dt_user_product_iap_revenue` only sees completed purchases. `rp_IAPStart` fires when a player taps "buy" on a specific `product_id`, before success/fail/cancel is known — it's not a pure "offer shown" signal (no impression-level event exists for `daily_offers` in dbt today), but it does let us see checkout attempts that didn't convert, which the revenue-only charts above can't.

In [14]:
# hide-output
starts = bqc.load_or_query('daily_offers_starts', refresh_data)

This query will process 4.5 MB when run.
Estimated query cost: $0.00


In [15]:
starts_by_product = starts.groupby('product_id', as_index=False)['n_starts'].sum()

conversion = product_summary[[
    'product_id', 'offer_quarter', 'bundle_type', 'product_type',
    'product_price_group', 'price_point_usd', 'n_trans', 'usd_revenue',
]].merge(starts_by_product, on='product_id', how='left')
conversion['n_starts'] = conversion['n_starts'].fillna(0)
conversion['conversion_rate_pct'] = (conversion['n_trans'] / conversion['n_starts'].mask(conversion['n_starts'] == 0) * 100).round(1)

#conversion.sort_values('n_starts', ascending=False)

### Overall, by Offer Quarter

In [16]:
conv_by_quarter = conversion.groupby('offer_quarter', as_index=False).agg(
    n_starts=('n_starts', 'sum'),
    n_trans=('n_trans', 'sum'),
)
conv_by_quarter['conversion_rate_pct'] = (conv_by_quarter['n_trans'] / conv_by_quarter['n_starts'] * 100).round(1)

conv_by_quarter_long = conv_by_quarter.melt(
    id_vars=['offer_quarter', 'conversion_rate_pct'],
    value_vars=['n_starts', 'n_trans'],
    var_name='metric', value_name='count',
)
conv_by_quarter_long['metric'] = conv_by_quarter_long['metric'].map({'n_starts': 'Started', 'n_trans': 'Completed'})
conv_by_quarter_long['label_text'] = conv_by_quarter_long.apply(
    lambda r: f"{r['conversion_rate_pct']:.1f}% conv." if r['metric'] == 'Completed' else '', axis=1
)

fig = px.bar(
    conv_by_quarter_long, x='offer_quarter', y='count', color='metric', barmode='group',
    color_discrete_map={'Started': '#2a78d6', 'Completed': '#1baf7a'},
    text='label_text',
)
fig.update_traces(textposition='outside', hovertemplate='%{y:,.0f}<extra></extra>')
fig.update_layout(
    title='Checkout Starts vs Completed Purchases by Offer Quarter',
    xaxis=dict(title=None, type='category'),
    yaxis=dict(title='Count'),
    height=450, width=800,
    legend=dict(orientation='h', yanchor='bottom', y=-0.25, xanchor='center', x=0.5, title=None),
)
fig.show()

### By Price Tier and Offer Quarter

Where the overall number could hide it — do higher-priced tiers convert worse under 2026Q3 than they did under 2025Q4? Restricted to tier/quarter combinations with at least one recorded checkout start.

In [17]:
price_tier_order = ['0099_0499', '0599_0999', '1099_1999', '2099_4999', '5099_plus']

conv_by_tier = conversion.groupby(['offer_quarter', 'product_price_group'], as_index=False).agg(
    n_starts=('n_starts', 'sum'),
    n_trans=('n_trans', 'sum'),
)
conv_by_tier = conv_by_tier[conv_by_tier['n_starts'] > 0]
conv_by_tier['conversion_rate_pct'] = (conv_by_tier['n_trans'] / conv_by_tier['n_starts'] * 100).round(1)

quarter_colors = {'2025Q4': '#2a78d6', '2026Q3': '#eb6834'}

fig = px.bar(
    conv_by_tier, x='product_price_group', y='conversion_rate_pct', color='offer_quarter',
    barmode='group',
    color_discrete_map=quarter_colors,
    category_orders={'product_price_group': price_tier_order},
    custom_data=['n_starts', 'n_trans'],
)
fig.update_traces(
    hovertemplate='Conversion: %{y:.1f}%<br>Started: %{customdata[0]:,.0f}<br>Completed: %{customdata[1]:,.0f}<extra></extra>'
)
fig.update_layout(
    title='Checkout Conversion by Price Tier and Offer Quarter',
    xaxis=dict(title='Price Tier (USD)'),
    yaxis=dict(title='Conversion Rate', ticksuffix='%'),
    height=450, width=1000,
    legend=dict(orientation='h', yanchor='bottom', y=-0.3, xanchor='center', x=0.5, title=None),
)
fig.show()

NOTE: The product price groups (price tier) is not an entirely fair comparison as 2025Q4 offers have more price points by design, but it does give an idea of the overall segment performance not the offers particularly

### Checkout Conversion by Price Point

This includes every offer from both quarters, not just bundle/product-type families that carried over into 2026Q3 — still broken out by individual price point and by bundle/product-type so different offers aren't pooled into one tier average.

In [18]:
# hide_output
conv_compare = conversion[conversion['n_starts'] > 0].copy()
conv_compare['combined_dim'] = conv_compare['bundle_type'] + ' · ' + conv_compare['product_type']
conv_compare['price_label'] = conv_compare['price_point_usd'].astype(str)
conv_compare = conv_compare.sort_values('price_point_usd')

fig = px.bar(
    conv_compare, x='price_label', y='conversion_rate_pct', color='combined_dim', pattern_shape='offer_quarter',
    barmode='group',
    custom_data=['n_starts', 'n_trans', 'offer_quarter'],
)
fig.update_traces(
    hovertemplate=(
        'Conversion: %{y:.1f}%<br>Started: %{customdata[0]:,.0f}<br>'
        'Completed: %{customdata[1]:,.0f}<br>Quarter: %{customdata[2]}<extra></extra>'
    ),
)
fig.update_layout(
    title='Checkout Conversion by Price Point (2025Q4 vs 2026Q3)',
    xaxis=dict(title='Price Point (USD)', type='category'),
    yaxis=dict(title='Conversion Rate', ticksuffix='%'),
    height=550, width=1300,
    legend=dict(orientation='v', yanchor='top', y=1, xanchor='left', x=1.02, title=None),
)
#fig.show()

## Player Price Migration: Did Players Move to a Higher Price?

Comparing "same bundle, same price" across quarters mixes different days and different player populations — not a fair test. This instead tracks individual `RotatingIncPack` players: for each player who bought a given `bundle_type` both before and on/after the 2026Q3 launch (2026-08-26), compare their own last pre-launch price against their own first post-launch price. Same player, so the population is controlled by construction — only time changes.

In [19]:
launch_date = pd.Timestamp('2026-08-26').date()
migration_base = data[data['product_type'] == 'RotatingIncPack']

before_launch = migration_base[migration_base['dt'] < launch_date].sort_values('dt')
after_launch = migration_base[migration_base['dt'] >= launch_date].sort_values('dt')

baseline = before_launch.groupby(['user_id', 'bundle_type'], as_index=False)['price_point_usd'].last().rename(columns={'price_point_usd': 'baseline_price'})
post_first = after_launch.groupby(['user_id', 'bundle_type'], as_index=False)['price_point_usd'].first().rename(columns={'price_point_usd': 'post_price'})

migration = baseline.merge(post_first, on=['user_id', 'bundle_type'], how='inner')
migration['direction'] = pd.cut(
    migration['post_price'] - migration['baseline_price'],
    bins=[-float('inf'), -0.001, 0.001, float('inf')],
    labels=['Moved lower', 'Stuck same', 'Moved higher'],
)

#migration

In [20]:
migration_by_bundle = migration.groupby(['bundle_type', 'direction'], observed=True).size().reset_index(name='n_players')
migration_by_bundle['pct'] = migration_by_bundle.groupby('bundle_type')['n_players'].transform(lambda s: (s / s.sum() * 100).round(1))

direction_colors = {'Moved lower': '#1baf7a', 'Stuck same': '#2a78d6', 'Moved higher': '#eb6834'}

fig = px.bar(
    migration_by_bundle, x='bundle_type', y='pct', color='direction',
    color_discrete_map=direction_colors,
    category_orders={'direction': ['Moved lower', 'Stuck same', 'Moved higher']},
    custom_data=['n_players'],
)
fig.update_traces(hovertemplate='%{y:.1f}%<br>Players: %{customdata[0]:,.0f}<extra></extra>')
fig.update_layout(
    barmode='stack',
    title='Player Price Migration by Bundle Type (Before → After 2026Q3 Launch)',
    xaxis=dict(title=None),
    yaxis=dict(title='Share of Players', ticksuffix='%'),
    height=450, width=800,
    legend=dict(orientation='h', yanchor='bottom', y=-0.25, xanchor='center', x=0.5, title=None),
)
fig.show()

## Offer Value: Rewards per Dollar

2026Q3 was designed to move players to a higher price tier while also giving more rewards — the fair test of that isn't "same price, different quarter" (the previous section), it's "does the reward you get keep pace with what you pay." `rpp_lighthouse_products` (the IAP catalogue, joined by `product_id`) carries `product_contents_array` and `product_bonus_array` — item/currency grants per product, as `(id, cnt)` pairs. `total_value` sums every `cnt` across both arrays (a blunt unweighted count — gems and energy dominate it, cosmetic items barely register), and `value_per_dollar = total_value / price` is the resulting "how much stuff per dollar" metric.

In [21]:
# hide-output
rewards = bqc.load_or_query('daily_offers_rewards', refresh_data)

This query will process 29.8 MB when run.
Estimated query cost: $0.00


In [22]:
rewards['total_value'] = rewards['total_contents_cnt'].fillna(0) + rewards['total_bonus_cnt'].fillna(0)

value = product_summary[[
    'product_id', 'offer_quarter', 'bundle_type', 'product_type', 'product_price_group', 'price_point_usd', 'n_trans',
]].merge(rewards[['product_id', 'total_value']], on='product_id', how='left')
value['value_per_dollar'] = (value['total_value'] / value['price_point_usd']).round(1)

#value.sort_values(['bundle_type', 'price_point_usd'])

In [23]:
#value[['bundle_type', 'product_type', 'price_point_usd', 'offer_quarter', 'n_trans', 'total_value', 'value_per_dollar',]].sort_values(['bundle_type', 'product_type', 'price_point_usd', 'offer_quarter'])

### Value per Dollar by Price Tier and Offer Quarter

Same price-tier cut as the conversion chart above, so the two can be read side by side: did the tiers where 2026Q3 converted worse also deliver worse value per dollar? The answer is, not at the 1 to 1 comparison level but at the tier level they do due to the high value and wider range of price points, specially at the lower segment.

In [24]:
value['weighted'] = value['value_per_dollar'] * value['n_trans']
value_by_tier = value.groupby(['offer_quarter', 'product_price_group'], as_index=False).agg(
    weighted_sum=('weighted', 'sum'),
    n_trans=('n_trans', 'sum'),
    avg_price_point_usd=('price_point_usd', 'mean')
)
value_by_tier['value_per_dollar_avg'] = (value_by_tier['weighted_sum'] / value_by_tier['n_trans']).round(1)
value_by_tier = value_by_tier[value_by_tier['n_trans'] > 0]

fig = px.bar(
    value_by_tier, x='product_price_group', y='value_per_dollar_avg', color='offer_quarter',
    barmode='group',
    color_discrete_map=quarter_colors,
    category_orders={'product_price_group': price_tier_order},
    custom_data=['n_trans'],
)
fig.update_traces(
    hovertemplate='Value/$: %{y:,.0f}<br>Transactions: %{customdata[0]:,.0f}<extra></extra>'
)
fig.update_layout(
    title='Purchase-Weighted Value per Dollar by Price Tier and Offer Quarter',
    xaxis=dict(title='Price Tier (USD)'),
    yaxis=dict(title='Value per Dollar (reward units)'),
    height=450, width=1000,
    legend=dict(orientation='h', yanchor='bottom', y=-0.3, xanchor='center', x=0.5, title=None),
)
fig.show()

# Average price_point_usd by price tier and offer quarter

fig = px.bar(
    value_by_tier, x='product_price_group', y='avg_price_point_usd', color='offer_quarter',
    barmode='group',
    color_discrete_map=quarter_colors,
    category_orders={'product_price_group': price_tier_order},
    custom_data=['n_trans'],
)
fig.update_traces(
    hovertemplate='Avg. Price Point (USD): %{y:,.2f}<br>Transactions: %{customdata[0]:,.0f}<extra></extra>'
)
fig.update_layout(
    title='Average Price Point by Price Tier and Offer Quarter',
    xaxis=dict(title='Price Tier (USD)'),
    yaxis=dict(title='Average Price Point (USD)'),
    height=450, width=1000,
    legend=dict(orientation='h', yanchor='bottom', y=-0.3, xanchor='center', x=0.5, title=None),
)
fig.show()

NOTE: The product price groups (price tier) is not an entirely fair comparison as 2025Q4 offers have more price points by design, but it does give an idea of the overall segment performance not the offers particularly

### Value per Dollar by Price Point (2025Q4 vs 2026Q3)

Same rewards-per-dollar metric, plotted per price point with bundle type and product type as color and offer quarter as pattern. Restricted to the bundle/product-type families that actually exist in 2026Q3 (drops 2025Q4-only lines like `PartyBundle · Rotating` or the `BundleSmall/Medium/Large` NonPayer offers), so this shows the full 2025Q4 price ladder for each family next to where its 2026Q3 price points land on it and confirms reward per dollar for the new offers was at least the same value or slightly higher. 

In [25]:
value_compare = value[value['value_per_dollar'].notna()].copy()
value_compare['combined_dim'] = value_compare['bundle_type'] + ' · ' + value_compare['product_type']

# Only compare bundle/product-type families that exist in both quarters
q3_combos = value_compare.loc[value_compare['offer_quarter'] == '2026Q3', 'combined_dim'].unique()
value_compare = value_compare[value_compare['combined_dim'].isin(q3_combos)]
value_compare['price_label'] = value_compare['price_point_usd'].astype(str)
value_compare = value_compare.sort_values('price_point_usd')

fig = px.bar(
    value_compare, x='price_label', y='value_per_dollar', color='combined_dim', pattern_shape='offer_quarter',
    barmode='group',
    custom_data=['n_trans', 'total_value', 'offer_quarter'],
)
fig.update_traces(
    hovertemplate=(
        'Value/$: %{y:,.1f}<br>Total value: %{customdata[1]:,.0f}<br>'
        'Transactions: %{customdata[0]:,.0f}<br>Quarter: %{customdata[2]}<extra></extra>'
    ),
)
fig.update_layout(
    title='Value per Dollar by Price Point (2025Q4 vs 2026Q3)',
    xaxis=dict(title='Price Point (USD)', type='category'),
    yaxis=dict(title='Value per Dollar (reward units)'),
    height=500, width=1200,
    legend=dict(orientation='h', yanchor='bottom', y=-0.4, xanchor='center', x=0.5, title=None),
)
fig.show()

In [26]:
export_notebook_html(
# Exports the notebook to HTML for sharing and archival
    notebook_path='./daily_offers.ipynb',
    output_path='./daily_offers.html',
)

Saved to daily_offers.html


PosixPath('daily_offers.html')